# CNN demo (benchmark B2)

Trains the reported architecture (C48-RF127: six dilated convolutions, 48 channels, receptive field 127 cells) for a few epochs and evaluates it. A few epochs is enough to exercise the whole pipeline, not to reach the reported accuracy.

For the full training budget and the receptive-field ablation this code also runs, see the main [README's Training section](../README.md#training-a-model).

Requires `GW_DATA` set before launching Jupyter, or the data placed under `data/` at the repository root.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path.cwd().parent  # this notebook lives in notebooks/
os.environ.setdefault("GW_DATA", str(REPO / "data"))

sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "src" / "cnn"))
import cnn_surrogate as cnn

print("repository:", REPO)
print("data root: ", os.environ["GW_DATA"])

`cnn_surrogate.train()` resolves the benchmark's snapshots through `GW_DATA` on its own, writes `test_predictions/` and `rollout_predictions/` under `out_dir`, and returns the trained model and test-set metrics.

In [ ]:
OUT_DIR = REPO / "out" / "cnn_demo"

cnn.train(seed=42, epochs=3, benchmark="b2",
          arch="dilated", channels=12, out_dir=str(OUT_DIR))

The reported model uses six dilated layers at 48 channels rather than the 12 above -- that configuration is trained through the receptive-field driver, not `cnn_surrogate.py`'s own `--arch` flag; see the main README.

## Conservation diagnostics on the test predictions

In [ ]:
subprocess.run(
    ["python", str(REPO / "src" / "diagnostics" / "evaluate_b2.py"),
     "--prediction-dir", str(OUT_DIR / "test_predictions"),
     "--output-dir", str(REPO / "out" / "cnn_diagnostics"),
     "--model-name", "CNN B2 demo",
     "--prediction-mode", "one-step", "--no-plots"],
    check=True,
)

The reported CNN reaches R² > 0.9998 with a residual ratio near 138 and a mass-balance error near 62% of the well rate -- see the main README's results table.